In [ ]:
from Parts.imports import *
from Parts.data_loader import DatasetLoader, DatasetLoaderV2
from Parts.models import U_NET_VANILLA, U_NET_RESNET, U_NET_RESNET_ATTENTION, U_NET_PLUS_PLUS, TRANS_U_NET
from Parts.training_loop import TrainingLoop, TrainingLoopAdvanced, SaveState, EarlyStopping
from Parts.losses import DiceLoss, Mixed_Dice_Sigmoid, IntersectionOverUnion
import tqdm

DEBUG = False

In [ ]:
PATH_TO_SAVED = r"model_weights/U_NET_RESNET.torch"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

df = torch.load(f=PATH_TO_SAVED, map_location=DEVICE)

In [ ]:
model = U_NET_RESNET()
model_weights = df['model_state_dict']
optim_weights = df['optimizer_state_dict']

model.load_state_dict(state_dict=model_weights)

In [ ]:
loss_mixed = Mixed_Dice_Sigmoid()
loss_dice = DiceLoss()
loss_iou = IntersectionOverUnion()

In [ ]:
data_test = r"C:\Users\itizs\Downloads\archive(1)\brisc2025\segmentation_task\test\images"
label_test = r"C:\Users\itizs\Downloads\archive(1)\brisc2025\segmentation_task\test\masks"
test_data = DatasetLoaderV2(data_dir=data_test, label_dir=label_test, dim=(256, 256), transformations=None)
test_data_loader = DataLoader(dataset=test_data, batch_size=8, shuffle=False, num_workers=0)

In [ ]:
model.to(DEVICE)
model.eval()

In [ ]:
records = []
with torch.no_grad():
    bar = tqdm.tqdm(total= len(test_data_loader))
    for i, (image, label) in enumerate(test_data_loader):
        if i == 25:
            break
        # image.unsqueeze(0)
        # label.unsqueeze(0)
        image.to(DEVICE)
        label.to(DEVICE)
        
        prediction = model(image)

        dice = loss_dice(label, prediction)
        iou = loss_iou(label, prediction)
        mix = loss_mixed(label, prediction)
        records.append({ 'index' : i, 
                    'dice_loss' : dice.item(),
                    # 'iou_loss' : iou.item(),
                    'mix_loss' : mix.item()})
        bar.update(1)
record = pandas.DataFrame(records)

In [ ]:
def show(idx, dataloader):
    image, label = dataloader[idx]
    
    image = image.unsqueeze(0) # need batch dim
    image.to(DEVICE)
    label.to(DEVICE)

    prediction = model(image)

    dice = loss_dice(label, prediction)
    iou = loss_iou(label, prediction)
    mix = loss_mixed(label, prediction)

    prediction = prediction.sigmoid().detach().cpu().squeeze().numpy()

    fig, axes = plt.subplots(1, 3)
    axes[0].imshow(label.squeeze(), cmap='gray')
    axes[0].set_title("actual mask")
    axes[0].axis('off')

    axes[1].imshow(prediction, cmap='jet')
    axes[1].set_title("Predicted Segmentation")
    axes[1].axis('off')

    axes[2].imshow(image.detach().cpu().squeeze().sigmoid().numpy().transpose(1, 2, 0))
    axes[2].set_title("Input Image")
    axes[2].axis('off')
show(9, test_data)

In [ ]:
record